In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set the default DPI for inline display and saved files to 300
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300
import warnings
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# pd.set_option("display.max_rows", None)
# # Optional: also show all columns
# pd.set_option("display.max_columns", None)
import seaborn as sns

sns.set_palette("tab10")
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("AIDS_Classification_50000.csv")

In [3]:
time_col = "time"
event_col = "infected"

x = df.drop(columns=[time_col, event_col])
t = df[time_col].values
e = df[event_col].values

In [4]:
# Optional: split data
X_train, X_test, e_train, e_test, t_train, t_test = train_test_split(
    x, e, t, test_size=0.2, random_state=42
)

In [5]:
cat_cols = [col for col in x.columns if x[col].nunique() < 10]
num_cols = [col for col in x.columns if col not in cat_cols]

In [6]:
# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        (
            "cat",
            OneHotEncoder(drop="first", handle_unknown="ignore"),
            cat_cols,
        ),
    ],
    remainder="passthrough",  # Keep any other columns unchanged (optional)
)

In [7]:
# Feature scaling

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

Hyperparameter Tuning for DeepSurv

In [9]:
# # Create a Sklearn-Compatible DeepSurv Wrapper
# import torch
# import torch.nn as nn
# from pycox.models import CoxPH
# from pycox.evaluation import concordance
# from sklearn.base import BaseEstimator, RegressorMixin
# from sklearn.utils.validation import check_X_y
# import numpy as np


# class DeepSurvEstimator(BaseEstimator, RegressorMixin):
#     def __init__(
#         self,
#         hidden_dims=[32, 32],
#         dropout=0.2,
#         batch_size=256,
#         epochs=100,
#         lr=0.01,
#         verbose=False,
#         random_state=42,
#     ):
#         self.hidden_dims = hidden_dims
#         self.dropout = dropout
#         self.batch_size = batch_size
#         self.epochs = epochs
#         self.lr = lr
#         self.verbose = verbose
#         self.random_state = random_state

#     def _build_net(self, in_features):
#         layers = []
#         prev_dim = in_features
#         for h_dim in self.hidden_dims:
#             layers.append(nn.Linear(prev_dim, h_dim))
#             layers.append(nn.ReLU())
#             layers.append(nn.Dropout(self.dropout))
#             prev_dim = h_dim
#         layers.append(nn.Linear(prev_dim, 1))
#         return nn.Sequential(*layers)

#     def fit(self, X, y):
#         # Handle structured array from sksurv
#         if hasattr(y, 'dtype') and y.dtype.names is not None:
#             e = y['event']
#             t = y['time']
#         else:
#             raise ValueError("y must be a structured array from sksurv.util.Surv")
        
#         # Rest of your fit logic...
#         X = X.astype(np.float32)
#         t = t.astype(np.float32)
#         e = e.astype(bool)

#         net = self._build_net(X.shape[1])
#         self.model_ = CoxPH(net)

#         # Set optimizer with current lr
#         self.model_.optimizer = torch.optim.Adam(
#             self.model_.net.parameters(), lr=self.lr
#         )

#         # Fit silently or verbosely
#         self.model_.fit(
#             input=X,
#             target=(t, e),
#             batch_size=self.batch_size,
#             epochs=self.epochs,
#             verbose=self.verbose,
#         )
#         return self

#     def predict(self, X):
#         """Return risk score (log partial hazard). Higher = worse."""
#         X = X.astype(np.float32)
#         return self.model_.predict(X)

#     def score(self, X, y):
#         """Return concordance index (higher is better)."""
#         t, e = y
#         risk = self.predict(X)
#         # concordance_td returns C-index; sklearn maximizes score
#         ctd = concordance(df_time=t, df_event=e, df_pred=risk)
#         return ctd

In [10]:
# # Define Hyperparameter Grid
# param_grid = {
#     "hidden_dims": [[32], [64], [32, 32], [64, 32]],
#     "dropout": [0.1, 0.2, 0.3],
#     "lr": [0.001, 0.01, 0.1],
#     "batch_size": [128, 256],
#     "epochs": [50, 100],
# }

In [26]:
from sksurv.util import Surv

# e_train: event indicator (0/1 or bool), shape (n,)
# t_train: time, shape (n,)
y_train = Surv.from_arrays(event=e_train, time=t_train)
y_test  = Surv.from_arrays(event=e_test, time=t_test)

In [12]:
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import make_scorer

# y_train = Surv.from_arrays(event=e_train, time=t_train)

# # Define cv (now safe to use StratifiedKFold on event)
# from sklearn.model_selection import StratifiedKFold

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# def survival_concordance_score(y_true, y_pred):
#     """Robust C-index scorer that never returns NaN."""
#     import numpy as np
#     from sksurv.metrics import concordance_index_censored
#     y_pred = np.ravel(y_pred)  # converts (n,) or (n,1) → (n,)
#     e = np.ravel(y_true["event"])
#     t = np.ravel(y_true["time"])

#     # Extract and sanitize
#     e = np.asarray(y_true["event"]).astype(bool)
#     t = np.asarray(y_true["time"], dtype=np.float64)
#     y_pred = np.asarray(y_pred, dtype=np.float64)

#     y_pred = np.ravel(y_pred)  # converts (n,) or (n,1) → (n,)
#     e = np.ravel(y_true["event"])
#     t = np.ravel(y_true["time"])
    
#     # Remove invalid entries
#     valid = (t > 0) & np.isfinite(t) & np.isfinite(y_pred)
#     e, t, y_pred = e[valid], t[valid], y_pred[valid]

#     if len(t) == 0:
#         return 0.5

#     # Check for all censored or all events
#     n_events = e.sum()
#     if n_events == 0 or n_events == len(e):
#         return 0.5

#     # Check for constant predictions
#     if np.std(y_pred) < 1e-8:
#         return 0.5

#     try:
#         c_index = concordance_index_censored(e, t, y_pred)[0]
#         return float(c_index) if not np.isnan(c_index) else 0.5
#     except:
#         return 0.5


# survival_scorer = make_scorer(survival_concordance_score, greater_is_better=True)

# # Fit
# random_search = RandomizedSearchCV(
#     estimator=DeepSurvEstimator(random_state=42),
#     param_distributions=param_grid,
#     n_iter=20,
#     cv=cv,
#     scoring=survival_scorer,
#     n_jobs=1,
#     verbose=5,
#     random_state=42
# )
# random_search.fit(X_train, y_train)

# # Best results
# print("Best C-index:", random_search.best_score_)
# print("Best params:", random_search.best_params_)

Best C-index: 0.6661265647272843
Best params: {'lr': 0.001, 'hidden_dims': [32], 'epochs': 100, 'dropout': 0.3, 'batch_size': 256}

In [13]:
best_params = {
    "lr": 0.001,
    "hidden_dims": [32],
    "epochs": 100,
    "dropout": 0.3,
    "batch_size": 256,
}

Define and Train DeepSurv

In [18]:
import torch
import torch.nn as nn
from pycox.models import CoxPH


# Define net using best hidden_dims and dropout
class BestDeepSurvNet(nn.Module):
    def __init__(self, in_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev_dim = in_features
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# Build model
net = BestDeepSurvNet(
    in_features=X_train.shape[1],
    hidden_dims=best_params["hidden_dims"],
    dropout=best_params["dropout"],
)

model = CoxPH(net)

# Set optimizer with best lr
model.optimizer = torch.optim.Adam(net.parameters(), lr=best_params["lr"])

# Fit on full data
model.fit(
    input=X_train,
    target=(t_train,e_train),  # pycox expects (time, event)
    batch_size=best_params["batch_size"],
    epochs=best_params["epochs"],
    verbose=True,
)

# Predict
risk_test = model.predict(X_test)

0:	[0s / 0s],		train_loss: 4.5674
1:	[0s / 0s],		train_loss: 4.5223
2:	[0s / 0s],		train_loss: 4.5129
3:	[0s / 0s],		train_loss: 4.5089
4:	[0s / 0s],		train_loss: 4.5074
5:	[0s / 1s],		train_loss: 4.5045
6:	[0s / 1s],		train_loss: 4.5011
7:	[0s / 1s],		train_loss: 4.4995
8:	[0s / 1s],		train_loss: 4.5025
9:	[0s / 1s],		train_loss: 4.4969
10:	[0s / 2s],		train_loss: 4.4979
11:	[0s / 2s],		train_loss: 4.4977
12:	[0s / 2s],		train_loss: 4.4984
13:	[0s / 2s],		train_loss: 4.4961
14:	[0s / 2s],		train_loss: 4.4945
15:	[0s / 3s],		train_loss: 4.4976
16:	[0s / 3s],		train_loss: 4.4912
17:	[0s / 3s],		train_loss: 4.4922
18:	[0s / 3s],		train_loss: 4.4929
19:	[0s / 3s],		train_loss: 4.4930
20:	[0s / 4s],		train_loss: 4.4947
21:	[0s / 4s],		train_loss: 4.4972
22:	[0s / 4s],		train_loss: 4.4943
23:	[0s / 4s],		train_loss: 4.4935
24:	[0s / 4s],		train_loss: 4.4906
25:	[0s / 5s],		train_loss: 4.4926
26:	[0s / 5s],		train_loss: 4.4913
27:	[0s / 5s],		train_loss: 4.4902
28:	[0s / 5s],		train_loss: 4.

In [19]:
# from pycox.models import CoxPH
# from pycox.evaluation import EvalSurv
# import torch
# import torch.nn as nn

# class DeepSurvNet(nn.Module):
#     def __init__(self, in_features):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(in_features, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 1)
#         )

#     def forward(self, x):
#         return self.net(x)


# net = DeepSurvNet(X_train.shape[1])
# model = CoxPH(net)

# # ⚠️ Set optimizer with desired learning rate BEFORE fit
# model.optimizer = torch.optim.Adam(model.net.parameters(), lr=0.005)

# # ✅ Fit without 'lr' argument
# model.fit(
#     input=X_train, target=(t_train, e_train), batch_size=256, epochs=100, verbose=True
# )

Evaluate and Predict

In [20]:
from pycox.evaluation import EvalSurv

In [21]:
model.compute_baseline_hazards()

# Predict survival function on test set
# ⚠️ Requires calling `predict_surv_df` — this estimates survival using Breslow baseline
surv = model.predict_surv_df(X_test)  # returns pd.DataFrame: rows=time, cols=samples

# Now create EvalSurv correctly
eval_surv = EvalSurv(
    surv=surv, durations=t_test, events=e_test, censor_surv="km"  # or another estimator
)

c_index = eval_surv.concordance_td()

In [22]:
c_index

0.6611139325030861

In [24]:
eval_surv.integrated_brier_score(t_test)

0.8422185870717676